# Audio Segmentation Pipeline

This Colab provides a streamlined pipeline for extracting labeled audio segments from Google Cloud Storage (GCS) using JSONL annotations and exporting them back to GCS.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/watch-duty/radio-transcription/blob/main/model/colabs/common/chirp_and_gemini_segment_audio.ipynb)

### Core Functions
1. **Manifest Retrieval**: Fetches target file lists from GCS. Only annotated files are processed.
2. **Local Audio Caching**: Downloads source audio into a local cache. On subsequent runs, it skips downloads if the file is already present locally.
3. **Ground Truth Slicing**: Extract audio segments as defined by the start/end timestamps in the manifest. Optionally adds silence to start and end of audio.
4. **Automated Export**:
    * Uploads processed FLAC segments to GCS organized by `example_id`.
    * Generates and uploads a `batch_manifest.jsonl` containing metadata for all segments (GCS paths, offsets, and durations) to facilitate downstream ASR evaluation.

In [ ]:
# @title Install dependencies
%pip install -q --upgrade \
    loguru \
    soundfile \
    ffmpeg-python

In [ ]:
# @title Imports and environment configuration
import json
import sys
from pathlib import Path
from urllib.parse import urlparse

import ffmpeg
from loguru import logger

# @markdown ### User Configuration
# @markdown Please enter your GCP project and bucket details below:
GCP_PROJECT_ID = ""  # @param {type:"string"}
GCS_BUCKET = ""  # @param {type:"string"}
PROJECT_NAME = "one_hour_pilot"  # @param {type:"string"}
# @markdown Enable modifications (e.g. silence padding, future normalization):
AUDIO_PREPROCESSING = False  # @param {type:"boolean"}
# @markdown If False, skip segments that already exist in GCS:
OVERWRITE_EXISTING = False  # @param {type:"boolean"}

assert GCP_PROJECT_ID, "Please enter your GCP project ID"
assert GCS_BUCKET, "Please enter your GCS bucket name"
assert PROJECT_NAME, "PROJECT_NAME must be provided."

SOURCE_MANIFEST_URI = (
    f"gs://{GCS_BUCKET}/manifests/{PROJECT_NAME}_transcriptions.json"
)
GCS_OUTPUT_PREFIX = f"segmented_audio/{PROJECT_NAME}_audio"

LOCAL_BASE_PATH = "/content"
CACHE_DIR = f"{LOCAL_BASE_PATH}/raw_audio"
SEGMENTS_DIR = f"{LOCAL_BASE_PATH}/segments"
BATCH_MANIFEST_FILENAME = "batch_manifest.jsonl"

# Initialize loguru
logger.remove()
logger.add(sys.stderr, format="<level>{level}</level>: {message}");

In [ ]:
from google.colab import auth
from google.cloud import storage

# @title Authentication and client initialization
auth.authenticate_user()
!gcloud config set project {GCP_PROJECT_ID} --quiet

# Initialize GCS Client
gcs_client = storage.Client(project=GCP_PROJECT_ID)

In [ ]:
# @title Helper functions
import base64
import io
import json
from pathlib import Path
from urllib.parse import urlparse
from IPython.display import HTML, display
import numpy as np
import scipy.signal
import soundfile as sf


def ensure_local_gcs_audio(gcs_uri: str) -> str:
    """Ensures the audio file from GCS is available locally in CACHE_DIR."""
    parsed = urlparse(gcs_uri)
    bucket_name = parsed.netloc
    blob_name = parsed.path.lstrip("/")

    filename = Path(blob_name).name
    local_path = Path(CACHE_DIR) / filename

    # Ensure the parent directory exists before attempting to download
    local_path.parent.mkdir(parents=True, exist_ok=True)

    if not local_path.exists():
        logger.info(f"Downloading {filename} from GCS...")
        bucket = gcs_client.bucket(bucket_name)
        blob = bucket.blob(blob_name)
        blob.download_to_filename(str(local_path))
    return str(local_path)


def cleanup_gcs_output(output_prefix: str) -> None:
    """Deletes existing blobs in the output directory for a clean run."""
    bucket = gcs_client.bucket(GCS_BUCKET)
    blobs = list(bucket.list_blobs(prefix=output_prefix))
    if blobs:
        logger.info(
            f"Cleaning existing files from gs://{GCS_BUCKET}/{output_prefix}"
        )
        bucket.delete_blobs(blobs)


def run_pilot_pipeline() -> None:
    """Orchestrates extraction, preserving native audio specs. Fails fast on errors."""
    Path(CACHE_DIR).mkdir(parents=True, exist_ok=True)
    Path(SEGMENTS_DIR).mkdir(parents=True, exist_ok=True)

    # 1. Determine Output Prefix
    # If preprocessing is OFF, we label as _raw. Otherwise, use base path.
    current_output_prefix = (
        GCS_OUTPUT_PREFIX if AUDIO_PREPROCESSING else f"{GCS_OUTPUT_PREFIX}_raw"
    )

    if OVERWRITE_EXISTING:
        cleanup_gcs_output(current_output_prefix)

    # 2. Load the JSON or JSONL manifest from GCS
    parsed_manifest = urlparse(SOURCE_MANIFEST_URI)
    m_bucket = gcs_client.bucket(parsed_manifest.netloc)
    m_blob = m_bucket.blob(parsed_manifest.path.lstrip("/"))
    content = m_blob.download_as_text()

    manifest_data = []
    manifest_file_extension = Path(parsed_manifest.path).suffix.lower()

    if manifest_file_extension == ".json":
        manifest_data = json.loads(content)
        logger.info(f"Loaded JSON manifest with {len(manifest_data)} entries.")
    elif manifest_file_extension == ".jsonl":
        manifest_data = [
            json.loads(line) for line in content.strip().split("\n")
        ]
        logger.info(f"Loaded JSONL manifest with {len(manifest_data)} entries.")
    else:
        raise ValueError(
            f"Unsupported manifest file type: {manifest_file_extension}. Must be .json or .jsonl"
        )

    files_to_process = {}
    for entry in manifest_data:
        path = entry["audio_filepath"]
        if path not in files_to_process:
            files_to_process[path] = []
        files_to_process[path].append(entry)

    output_bucket = gcs_client.bucket(GCS_BUCKET)
    final_manifest_entries = []

    # 3. Process segments
    for gcs_audio_path, segments in files_to_process.items():
        local_src_path = None  # Lazy load only if needed
        example_id = Path(gcs_audio_path).stem

        for i, seg in enumerate(segments):
            seg_id = f"{i:03d}"
            transcription = seg.get("text", "")
            if not transcription:
                logger.info(
                    f"Skipping because no text. Of category: {seg.get('category', '')}"
                )
                continue

            start_s = seg["offset"]
            duration_s = seg["duration"]
            filename = f"{example_id}__seg{seg_id}.flac"
            blob_name = f"{current_output_prefix}/{example_id}/{filename}"
            output_blob = output_bucket.blob(blob_name)

            total_duration = (
                duration_s + 3 if AUDIO_PREPROCESSING else duration_s
            )

            # Skip if file exists and we aren't overwriting
            if not OVERWRITE_EXISTING and output_blob.exists():
                final_manifest_entries.append(
                    {
                        "audio_filepath": f"gs://{GCS_BUCKET}/{blob_name}",
                        "example_id": example_id,
                        "offset": start_s,
                        "duration": total_duration,
                        "segment_id": seg_id,
                        "text": transcription,
                    }
                )
                continue

            # Lazy load source audio and probe specs (fidelity preserved)
            if local_src_path is None:
                local_src_path = ensure_local_gcs_audio(gcs_audio_path)
                probe = ffmpeg.probe(local_src_path)
                audio_stream = next(
                    (s for s in probe["streams"] if s["codec_type"] == "audio"),
                    None,
                )
                if not audio_stream:
                    raise ValueError(f"No audio found in {gcs_audio_path}")
                native_sr = int(audio_stream["sample_rate"])
                native_channels = int(audio_stream["channels"])
                logger.info(
                    f"Processing {example_id} ({native_sr}Hz, {native_channels}ch)"
                )

            # ADDED: Per-segment progress logging
            logger.info(
                f"  > Slicing segment {seg_id} (offset: {start_s}s, duration: {duration_s}s)"
            )

            local_slice_path = Path(SEGMENTS_DIR) / filename

            # Build FFmpeg chain
            input_stream = ffmpeg.input(
                local_src_path, ss=start_s, t=duration_s
            )

            if AUDIO_PREPROCESSING:
                # Generate silence matching native channel layout
                layout = "mono" if native_channels == 1 else "stereo"
                silence_pre = ffmpeg.input(
                    f"anullsrc=r={native_sr}:cl={layout}", f="lavfi", t=1
                )
                silence_post = ffmpeg.input(
                    f"anullsrc=r={native_sr}:cl={layout}", f="lavfi", t=2
                )
                processed_stream = ffmpeg.concat(
                    silence_pre, input_stream, silence_post, v=0, a=1
                )
            else:
                processed_stream = input_stream

            # Final output encoding
            out = ffmpeg.output(processed_stream, str(local_slice_path))
            ffmpeg.run(out, overwrite_output=True, quiet=True)

            # Upload to GCS
            output_blob.upload_from_filename(str(local_slice_path))
            # Clean up local slice file after upload
            local_slice_path.unlink()

            final_manifest_entries.append(
                {
                    "audio_filepath": f"gs://{GCS_BUCKET}/{blob_name}",
                    "example_id": example_id,
                    "offset": start_s,
                    "duration": total_duration,
                    "segment_id": seg_id,
                    "text": transcription,
                }
            )

        # Clean up local source audio file after all its segments are processed
        if local_src_path and Path(local_src_path).exists():
            Path(local_src_path).unlink()

    # 4. Final Manifest
    local_manifest = Path(LOCAL_BASE_PATH) / BATCH_MANIFEST_FILENAME
    with open(local_manifest, "w") as f:
        for entry in final_manifest_entries:
            f.write(json.dumps(entry) + "\n")

    output_bucket.blob(
        f"{current_output_prefix}/{BATCH_MANIFEST_FILENAME}"
    ).upload_from_filename(str(local_manifest))
    logger.info(
        f"Done. {len(final_manifest_entries)} entries in manifest for {current_output_prefix}."
    )


def merge_segments(segments):
    """Merges consecutive segments that have text transcripts."""
    # Sort by offset to ensure sequential processing
    segments = sorted(segments, key=lambda x: x["offset"])
    merged = []
    current_merge = None

    for seg in segments:
        has_text = bool(seg.get("text", "").strip())
        category = seg.get("category", "").strip().lower()
        is_unintelligible = category == "unintelligible"

        if has_text and not is_unintelligible:
            if current_merge is None:
                current_merge = {
                    "audio_filepath": seg["audio_filepath"],
                    "example_id": seg.get(
                        "example_id", Path(seg["audio_filepath"]).stem
                    ),
                    "offset": seg["offset"],
                    "duration": seg["duration"],
                    "segment_id": seg.get("segment_id", "merged"),
                    "text": seg["text"],
                    "segments_merged": 1,
                }
            else:
                # Extend current merged segment
                end_time = max(
                    current_merge["offset"] + current_merge["duration"],
                    seg["offset"] + seg["duration"],
                )
                current_merge["duration"] = end_time - current_merge["offset"]
                current_merge["text"] += " " + seg["text"]
                current_merge["segments_merged"] += 1
        else:
            # Unintelligible segment breaks the consecutive merge chain
            if current_merge is not None:
                merged.append(current_merge)
                current_merge = None

    if current_merge is not None:
        merged.append(current_merge)

    return merged


def simulate_segment_merging(manifest_data):
    """Simulates merging neighboring transcribed segments without unintelligible sections."""
    files_to_process = {}
    for entry in manifest_data:
        path = entry["audio_filepath"]
        if path not in files_to_process:
            files_to_process[path] = []
        files_to_process[path].append(entry)

    all_merged_segments = []
    for path, segments in files_to_process.items():
        merged = merge_segments(segments)
        all_merged_segments.extend(merged)

    logger.info(
        f"Simulated merging: {len(manifest_data)} original segments -> {len(all_merged_segments)} merged segments."
    )
    return all_merged_segments


def visualize_audio_with_segments(
    audio_array: np.ndarray,
    segments: list,
    target_sr: int = 16000,
    container_id: str = "waveform",
    wave_color: str = "tomato",
    progress_color: str = "firebrick",
    title_text: str = "Audio Configuration",
) -> None:
    """Creates interactable waveform charts representing slice regions natively via WaveSurfer.js."""
    if audio_array.ndim > 1 and audio_array.shape[1] > 1:
        audio_array = np.mean(audio_array, axis=1)
    elif audio_array.ndim == 2 and audio_array.shape[1] == 1:
        audio_array = audio_array.flatten()

    wav_io = io.BytesIO()
    sf.write(wav_io, audio_array, target_sr, format="WAV", subtype="PCM_16")
    wav_io.seek(0)
    audio_b64 = base64.b64encode(wav_io.read()).decode("utf-8")

    regions_js = "".join(
        [
            f"wsRegions.addRegion({{start: {seg['start']}, end: {seg['end']}, content: '{seg['label']}', color: '{seg['color']}'}});\n"
            for seg in segments
        ]
    )

    html_code = f"""
    <div id='{container_id}' style='margin-top: 20px; border: 1px solid #ddd; border-radius: 4px;'></div>
    <div id='timeline-{container_id}'></div>
    <div style='margin-top: 10px; display: flex; align-items: center; gap: 15px;'>
        <button id='btn-play-{container_id}' style='padding: 8px 16px; cursor: pointer;'>Play / Pause</button>
        <span style='font-family: sans-serif; color: #555; font-size: 14px;'>Zoom: <input type="range" id="zoom-{container_id}" min="10" max="1000" value="10" style="width: 150px; vertical-align: middle;"></span>
        <span style='font-family: sans-serif; color: #555; margin-left: auto;'>{title_text} ({len(segments)} segments)</span>
    </div>
    <script type='module'>
        import WaveSurfer from 'https://unpkg.com/wavesurfer.js@7/dist/wavesurfer.esm.js';
        import RegionsPlugin from 'https://unpkg.com/wavesurfer.js@7/dist/plugins/regions.esm.js';
        import TimelinePlugin from 'https://unpkg.com/wavesurfer.js@7/dist/plugins/timeline.esm.js';
        import HoverPlugin from 'https://unpkg.com/wavesurfer.js@7/dist/plugins/hover.esm.js';

        const ws = WaveSurfer.create({{
            container: '#{container_id}',
            waveColor: '{wave_color}',
            progressColor: '{progress_color}',
            height: 150,
            normalize: true,
            minPxPerSec: 10,
        }});

        const wsRegions = ws.registerPlugin(RegionsPlugin.create());
        const wsTimeline = ws.registerPlugin(TimelinePlugin.create({{
            container: '#timeline-{container_id}',
            height: 24,
            style: {{
                fontSize: '12px',
                color: '#666',
            }}
        }}));

        ws.registerPlugin(HoverPlugin.create({{
            lineColor: '#ff0000',
            lineWidth: 2,
            labelBackground: '#555',
            labelColor: '#fff',
            labelSize: '11px',
            formatTimeCallback: (sec) => sec.toFixed(3) + 's'
        }}));

        ws.load('data:audio/wav;base64,{audio_b64}');
        ws.on('decode', () => {{
            {regions_js}
            const slider = document.getElementById('zoom-{container_id}');
            slider.addEventListener('input', (e) => {{
                ws.zoom(e.target.valueAsNumber);
            }});
        }});
        document.getElementById('btn-play-{container_id}').onclick = () => ws.playPause();
    </script>
    """
    display(HTML(html_code))


In [ ]:
# @title Create the pilot segments and manifest file
run_pilot_pipeline()

In [ ]:
# @title Visualize Original and Merged Segments Together
import soundfile as sf
from IPython.display import Audio, display, HTML
import json
from urllib.parse import urlparse
from pathlib import Path
import numpy as np
import io
import base64
import scipy.signal

# 1. Fetch the manifest data
parsed_manifest = urlparse(SOURCE_MANIFEST_URI)
m_bucket = gcs_client.bucket(parsed_manifest.netloc)
m_blob = m_bucket.blob(parsed_manifest.path.lstrip("/"))
content = m_blob.download_as_text()

# Robustly parse manifest data
try:
    manifest_data = json.loads(content)
except json.JSONDecodeError as e:
    if "Extra data" in str(e):
        manifest_data = [
            json.loads(line)
            for line in content.strip().split("\n")
            if line.strip()
        ]
    else:
        raise e

# 2. Simulate the merged segments
merged_segments = simulate_segment_merging(manifest_data)

# --- Determine the audio source to use for visualization ---
audio_for_visualization = None
sr_for_visualization = 16000
local_audio_cleanup_path = None

if not merged_segments:
    print("No merged segments found to visualize.")
else:
    original_gcs_audio_path = merged_segments[0]["audio_filepath"]
    original_audio_filename = Path(original_gcs_audio_path).name

    if "final_audio" in locals():
        audio_for_visualization = final_audio
        print(
            f"Using processed audio (from VAD pipeline) for visualization of {original_audio_filename}"
        )
    else:
        print(
            f"VAD pipeline not run. Downloading original raw audio from GCS for {original_audio_filename}"
        )
        full_local_src_path = ensure_local_gcs_audio(original_gcs_audio_path)
        raw_audio_data, raw_sr = sf.read(full_local_src_path)
        local_audio_cleanup_path = full_local_src_path

        # Resample if necessary
        if raw_sr != sr_for_visualization:
            print(
                f"  Resampling original audio from {raw_sr}Hz to {sr_for_visualization}Hz..."
            )
            num_samples_resampled = int(
                len(raw_audio_data) * sr_for_visualization / raw_sr
            )
            audio_for_visualization = scipy.signal.resample(
                raw_audio_data, num_samples_resampled
            )
        else:
            audio_for_visualization = raw_audio_data

    # Ensure audio_for_visualization is set before proceeding
    if audio_for_visualization is not None:
        segments_to_visualize = []

        # 1. Add Simulated Merged Segments FIRST (so they are in the background)
        merged_segments_for_display = [
            seg
            for seg in merged_segments
            if seg["audio_filepath"] == original_gcs_audio_path
        ]

        for i, seg_data in enumerate(merged_segments_for_display):
            start_time = seg_data["offset"]
            end_time = seg_data["offset"] + seg_data["duration"]

            text_preview = seg_data.get("text", "").strip()
            label = f"MERGED [{i}]"
            if text_preview:
                label += f": {text_preview[:15]}..."

            segments_to_visualize.append(
                {
                    "start": start_time,
                    "end": end_time,
                    "color": "rgba(0, 0, 255, 0.2)",  # Light Blue for merged blocks
                    "label": label,
                }
            )

        # 2. Add Original Segments SECOND (so they are drawn on top)
        original_segments_for_display = [
            seg
            for seg in manifest_data
            if seg["audio_filepath"] == original_gcs_audio_path
        ]
        original_segments_for_display.sort(key=lambda x: x["offset"])

        # Define color mapping for original categories
        category_colors = {
            "transcription": "rgba(50, 205, 50, 0.5)",  # Green
            "unintelligible": "rgba(255, 0, 0, 0.5)",  # Red
            "pii/id": "rgba(138, 43, 226, 0.5)",  # Purple
            "dtmf": "rgba(255, 165, 0, 0.5)",  # Orange
        }

        for seg_data in original_segments_for_display:
            start_time = seg_data["offset"]
            end_time = seg_data["offset"] + seg_data["duration"]

            cat = seg_data.get("category", "").lower()
            text_preview = seg_data.get("text", "").strip()

            # Determine color and label based on category
            if cat == "transcription" and text_preview:
                color = category_colors["transcription"]
                label = f"Orig: {text_preview[:15]}..."
            else:
                color = category_colors.get(
                    cat, "rgba(128, 128, 128, 0.5)"
                )  # Gray fallback
                label = f"Orig {cat.upper()}" if cat else "Orig UNKNOWN"
                if text_preview:
                    label += f": {text_preview[:15]}..."

            segments_to_visualize.append(
                {
                    "start": start_time,
                    "end": end_time,
                    "color": color,
                    "label": label,
                }
            )

        unique_container_id = f"waveform-{Path(original_gcs_audio_path).stem}"

        print(
            "Visualizing Original segments overlaid on top of Simulated Merged blocks..."
        )
        visualize_audio_with_segments(
            audio_array=audio_for_visualization,
            segments=segments_to_visualize,
            target_sr=sr_for_visualization,
            container_id=unique_container_id,
            title_text=f"Overlay View for {original_audio_filename} (Blue background = Merged)",
        )

        print("\n### Play Audio Source:")
        display(Audio(audio_for_visualization, rate=sr_for_visualization))

        # Clean up
        if local_audio_cleanup_path and Path(local_audio_cleanup_path).exists():
            Path(local_audio_cleanup_path).unlink()
            print(f"Cleaned up temporary local file: {local_audio_cleanup_path}")
    else:
        print("Failed to prepare audio for visualization.")
